# Stable Diffusion 実践的教材

このノートブックでは、Stable Diffusion を「画像生成モデルとして使える」だけでなく、実験条件を管理しながら品質・速度・再現性を調整できるようになることを目標にします。

扱う内容:

- Diffusion model と Stable Diffusion の直感
- Hugging Face Diffusers による text-to-image
- seed、steps、guidance scale、scheduler の比較
- negative prompt とプロンプト設計
- img2img と inpainting による編集
- GPU メモリ対策と実務での設定管理
- LoRA を使うときの基本形
- 生成結果の評価と失敗例の読み解き

注意: Stable Diffusion の実モデルは数GBの重みをダウンロードします。GPU がない環境でも構造を学べるように、最初は tiny pipeline で動作確認できる構成にしています。

## 0. 依存関係

この教材では `torch`, `diffusers`, `transformers`, `accelerate`, `safetensors`, `Pillow`, `matplotlib`, `pandas` を使います。

```bash
pip install torch diffusers transformers accelerate safetensors pillow matplotlib pandas
```

GPU で実行する場合は、環境に合った CUDA 対応版の PyTorch を入れてください。モデルによっては Hugging Face の利用規約への同意やログインが必要です。

In [ ]:
import gc
import math
import random
from dataclasses import dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFilter

try:
    import diffusers
    from diffusers import (
        DiffusionPipeline,
        StableDiffusionPipeline,
        StableDiffusionImg2ImgPipeline,
        StableDiffusionInpaintPipeline,
        EulerDiscreteScheduler,
        DPMSolverMultistepScheduler,
    )
except ImportError as e:
    raise ImportError('diffusers が見つかりません。先に依存関係セルの pip install を実行してください。') from e


def set_seed(seed: int = 42):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dtype = torch.float16 if device.type == 'cuda' else torch.float32
output_dir = Path('stable_diffusion_outputs')
output_dir.mkdir(exist_ok=True)

print('torch:', torch.__version__)
print('diffusers:', diffusers.__version__)
print('device:', device, 'dtype:', dtype)


## 1. Stable Diffusion の最小限の理解

Diffusion model は、画像に少しずつノイズを足していく過程を学び、その逆向きとして「ノイズから画像を復元する」モデルです。生成時はランダムノイズから始め、複数ステップかけてノイズを取り除きます。

Stable Diffusion は、この処理を画像ピクセル空間ではなく **latent space** で行います。画像を VAE で小さな潜在表現に圧縮し、その潜在表現上で U-Net がノイズ除去を行い、最後に VAE decoder で画像に戻します。

主な部品:

| 部品 | 役割 |
|---|---|
| Tokenizer / Text Encoder | prompt をベクトルに変換する |
| U-Net | 各ステップでノイズを予測する中心モデル |
| Scheduler | 何ステップで、どの強さでノイズを除くかを決める |
| VAE | 画像と latent を相互変換する |
| Safety checker | 一部 pipeline で安全性フィルタを行う |

実務では、モデル本体だけでなく `prompt`, `negative_prompt`, `seed`, `num_inference_steps`, `guidance_scale`, `scheduler`, `width`, `height` をセットで記録することが重要です。

## 2. 実験設定をまとめる

Stable Diffusion は同じ prompt でも seed や scheduler が変わると結果が大きく変わります。まず設定を dataclass にまとめ、後から比較しやすくします。

In [ ]:
@dataclass
class GenerationConfig:
    prompt: str
    negative_prompt: str = ''
    seed: int = 42
    steps: int = 25
    guidance_scale: float = 7.5
    width: int = 512
    height: int = 512


base_config = GenerationConfig(
    prompt='a cinematic photo of a small modern library interior, warm natural light, detailed shelves, realistic',
    negative_prompt='low quality, blurry, distorted, extra limbs, text, watermark',
    seed=123,
    steps=25,
    guidance_scale=7.5,
)
base_config


## 3. tiny pipeline で動作確認する

最初に軽量なテスト用 pipeline で、Diffusers の呼び出し方だけ確認します。このモデルは品質確認用ではありません。実務品質の画像は後の実モデルで確認します。

In [ ]:
tiny_model_id = 'hf-internal-testing/tiny-stable-diffusion-pipe'

tiny_pipe = DiffusionPipeline.from_pretrained(tiny_model_id, torch_dtype=dtype)
tiny_pipe = tiny_pipe.to(device)

generator = torch.Generator(device=device).manual_seed(0)
tiny_image = tiny_pipe(
    prompt='a photo of a red apple on a wooden table',
    num_inference_steps=2,
    generator=generator,
).images[0]

tiny_image.save(output_dir / 'tiny_smoke_test.png')
tiny_image


## 4. 実モデルを読み込む

次に Stable Diffusion の実モデルを読み込みます。デフォルトでは Stable Diffusion v1.5 を使います。GPU メモリが足りない場合は、`enable_attention_slicing()` や `enable_vae_slicing()` を使います。

モデルを変更したい場合は `model_id` を差し替えます。SDXL 系を使う場合は `StableDiffusionXLPipeline` など別 pipeline が必要になることがあります。

In [ ]:
model_id = 'stable-diffusion-v1-5/stable-diffusion-v1-5'

pipe = StableDiffusionPipeline.from_pretrained(
    model_id,
    torch_dtype=dtype,
    use_safetensors=True,
)
pipe = pipe.to(device)

if device.type == 'cuda':
    pipe.enable_attention_slicing()
    pipe.enable_vae_slicing()

pipe.scheduler.__class__.__name__


## 5. 生成関数を作る

毎回同じ形式で実験できるように、生成と保存を1つの関数にします。seed を固定すると、同じ環境・同じモデル・同じ設定では結果を再現しやすくなります。

In [ ]:
def make_generator(seed: int):
    return torch.Generator(device=device).manual_seed(seed)


def generate_image(pipe, cfg: GenerationConfig, name: str):
    generator = make_generator(cfg.seed)
    result = pipe(
        prompt=cfg.prompt,
        negative_prompt=cfg.negative_prompt,
        num_inference_steps=cfg.steps,
        guidance_scale=cfg.guidance_scale,
        width=cfg.width,
        height=cfg.height,
        generator=generator,
    )
    image = result.images[0]
    path = output_dir / name
    image.save(path)
    return image, path


image, path = generate_image(pipe, base_config, 'text2img_base.png')
print(path)
image


## 6. seed の影響を見る

seed は初期ノイズを決めます。prompt が同じでも seed が変わると構図・色・細部が変わります。実務では「良い seed」を保存しておくと、後から同じ方向性で調整できます。

In [ ]:
def show_grid(images, titles=None, cols=4, figsize=(12, 8)):
    rows = math.ceil(len(images) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = list(axes.flatten()) if hasattr(axes, 'flatten') else [axes]
    for i, ax in enumerate(axes):
        ax.axis('off')
        if i < len(images):
            ax.imshow(images[i])
            if titles:
                ax.set_title(titles[i], fontsize=10)
    plt.tight_layout()
    return fig


seed_images = []
seed_titles = []
for seed in [1, 2, 3, 4]:
    cfg = GenerationConfig(**{**base_config.__dict__, 'seed': seed, 'steps': 20})
    img, _ = generate_image(pipe, cfg, f'seed_{seed}.png')
    seed_images.append(img)
    seed_titles.append(f'seed={seed}')

show_grid(seed_images, seed_titles, cols=4, figsize=(12, 4))


## 7. guidance scale を比較する

`guidance_scale` は prompt への従いやすさを強める値です。低すぎると prompt から外れやすく、高すぎると不自然なコントラスト、硬い質感、破綻が増えることがあります。よく使う範囲はおおむね 5 から 12 です。

In [ ]:
guidance_images = []
guidance_titles = []
for scale in [3.0, 5.0, 7.5, 11.0]:
    cfg = GenerationConfig(**{**base_config.__dict__, 'guidance_scale': scale, 'steps': 25})
    img, _ = generate_image(pipe, cfg, f'guidance_{scale}.png')
    guidance_images.append(img)
    guidance_titles.append(f'CFG={scale}')

show_grid(guidance_images, guidance_titles, cols=4, figsize=(12, 4))


## 8. inference steps を比較する

`num_inference_steps` はノイズ除去の回数です。増やすほど遅くなります。一定以上増やしても品質が大きく伸びないことがあるため、実務では品質と時間のバランスを見ます。

In [ ]:
step_images = []
step_titles = []
for steps in [10, 20, 30, 50]:
    cfg = GenerationConfig(**{**base_config.__dict__, 'steps': steps, 'seed': 777})
    img, _ = generate_image(pipe, cfg, f'steps_{steps}.png')
    step_images.append(img)
    step_titles.append(f'steps={steps}')

show_grid(step_images, step_titles, cols=4, figsize=(12, 4))


## 9. scheduler を切り替える

Scheduler はノイズ除去の進め方を決めます。同じ prompt と seed でも scheduler を変えると結果が変わります。速さ重視、質感重視、少ない steps での安定性など、用途に合わせて比較します。

In [ ]:
scheduler_experiments = [
    ('default', pipe.scheduler),
    ('euler', EulerDiscreteScheduler.from_config(pipe.scheduler.config)),
    ('dpm_solver', DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)),
]

scheduler_images = []
scheduler_titles = []
original_scheduler = pipe.scheduler

for name, scheduler in scheduler_experiments:
    pipe.scheduler = scheduler
    cfg = GenerationConfig(**{**base_config.__dict__, 'seed': 314, 'steps': 25})
    img, _ = generate_image(pipe, cfg, f'scheduler_{name}.png')
    scheduler_images.append(img)
    scheduler_titles.append(name)

pipe.scheduler = original_scheduler
show_grid(scheduler_images, scheduler_titles, cols=3, figsize=(10, 4))


## 10. prompt を設計する

prompt は単語を詰め込むより、目的に沿って役割を分けると管理しやすくなります。

| 要素 | 例 |
|---|---|
| 主体 | small modern library interior |
| 構図 | wide angle, eye-level shot |
| 光 | warm natural light, soft shadows |
| 質感 | realistic, detailed wood shelves |
| スタイル | editorial architecture photography |
| 除外 | blurry, low quality, text, watermark |

実務では、最初に短い prompt で大きな方向性を決め、その後に光・構図・質感を足していく方が原因分析しやすくなります。

In [ ]:
prompt_variants = {
    'short': 'a modern library interior',
    'composition': 'a modern library interior, wide angle, eye-level shot',
    'lighting': 'a modern library interior, wide angle, warm natural light, soft shadows',
    'full': 'a modern library interior, wide angle, warm natural light, soft shadows, realistic architecture photography, detailed wood shelves',
}

prompt_images = []
prompt_titles = []
for name, prompt in prompt_variants.items():
    cfg = GenerationConfig(**{**base_config.__dict__, 'prompt': prompt, 'seed': 909, 'steps': 25})
    img, _ = generate_image(pipe, cfg, f'prompt_{name}.png')
    prompt_images.append(img)
    prompt_titles.append(name)

show_grid(prompt_images, prompt_titles, cols=4, figsize=(12, 4))


## 11. img2img で既存画像を変換する

img2img は、初期画像を latent に変換してからノイズを加え、prompt に沿って再生成します。`strength` が低いほど元画像を保ち、高いほど prompt に寄ります。

In [ ]:
def make_simple_room_image(size=512):
    image = Image.new('RGB', (size, size), '#d8d2c4')
    draw = ImageDraw.Draw(image)
    draw.rectangle([60, 120, 452, 430], fill='#c8bba5', outline='#6f6558', width=4)
    draw.rectangle([90, 160, 422, 390], fill='#efe9da')
    for x in range(120, 400, 70):
        draw.rectangle([x, 170, x + 35, 380], fill='#8f6b4a')
    draw.rectangle([150, 395, 360, 435], fill='#6f4e37')
    return image


init_image = make_simple_room_image()
init_image.save(output_dir / 'img2img_init.png')
init_image


In [ ]:
img2img_pipe = StableDiffusionImg2ImgPipeline(**pipe.components)
img2img_pipe = img2img_pipe.to(device)

img2img_prompt = 'a cozy modern reading room, realistic interior design photo, wooden shelves, warm lamps, high detail'
img2img_negative = 'low quality, blurry, distorted, text, watermark'

img2img_images = []
img2img_titles = []
for strength in [0.25, 0.55, 0.85]:
    result = img2img_pipe(
        prompt=img2img_prompt,
        negative_prompt=img2img_negative,
        image=init_image,
        strength=strength,
        guidance_scale=7.5,
        num_inference_steps=30,
        generator=make_generator(123),
    )
    img = result.images[0]
    img.save(output_dir / f'img2img_strength_{strength}.png')
    img2img_images.append(img)
    img2img_titles.append(f'strength={strength}')

show_grid([init_image] + img2img_images, ['init'] + img2img_titles, cols=4, figsize=(12, 4))


## 12. inpainting で一部だけ編集する

inpainting は、画像と mask を渡して、白い mask 領域だけを prompt に沿って生成します。商品写真の背景差し替え、不要物の除去、部分的なデザイン案出しに使えます。

In [ ]:
base_for_inpaint = image.resize((512, 512))
mask = Image.new('L', (512, 512), 0)
draw = ImageDraw.Draw(mask)
draw.ellipse([190, 300, 325, 435], fill=255)
mask = mask.filter(ImageFilter.GaussianBlur(radius=4))

base_for_inpaint.save(output_dir / 'inpaint_base.png')
mask.save(output_dir / 'inpaint_mask.png')

show_grid([base_for_inpaint, mask.convert('RGB')], ['base image', 'mask'], cols=2, figsize=(8, 4))


In [ ]:
inpaint_model_id = 'stable-diffusion-v1-5/stable-diffusion-inpainting'
inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
    inpaint_model_id,
    torch_dtype=dtype,
    use_safetensors=True,
)
inpaint_pipe = inpaint_pipe.to(device)

if device.type == 'cuda':
    inpaint_pipe.enable_attention_slicing()
    inpaint_pipe.enable_vae_slicing()

inpaint_result = inpaint_pipe(
    prompt='a comfortable leather reading chair, realistic, consistent lighting',
    negative_prompt='blurry, low quality, distorted, text, watermark',
    image=base_for_inpaint,
    mask_image=mask,
    num_inference_steps=30,
    guidance_scale=7.5,
    generator=make_generator(555),
).images[0]

inpaint_result.save(output_dir / 'inpaint_result.png')
show_grid([base_for_inpaint, mask.convert('RGB'), inpaint_result], ['base', 'mask', 'result'], cols=3, figsize=(12, 4))


## 13. LoRA を読み込む基本形

LoRA は、元モデル全体を更新せずに小さな追加重みで画風・人物・商品・ドメインを追加する方法です。ここでは実行しやすいようにコードの形だけ示します。実際に使う場合は、LoRA のライセンス、対象ベースモデル、推奨 trigger word を確認してください。

In [ ]:
# 例: LoRA を使う場合の基本形です。実行するには lora_repo_id を実在する LoRA に変更してください。
# lora_repo_id = 'your-org/your-lora-repo'
# pipe.load_lora_weights(lora_repo_id)
# pipe.set_adapters(['default'], adapter_weights=[0.7])
#
# cfg = GenerationConfig(
#     prompt='trigger_word, a product photo of a backpack, studio lighting',
#     negative_prompt='low quality, blurry, watermark',
#     seed=42,
#     steps=25,
#     guidance_scale=7.0,
# )
# image, path = generate_image(pipe, cfg, 'lora_example.png')
# image


## 14. GPU メモリ対策

Stable Diffusion は GPU メモリを多く使います。メモリ不足時は次の順に試します。

| 対策 | 効果 | 注意 |
|---|---|---|
| `torch.float16` | メモリ削減・高速化 | 主に GPU 向け |
| `enable_attention_slicing()` | attention のメモリ削減 | 少し遅くなる |
| `enable_vae_slicing()` | VAE のメモリ削減 | 大きい画像で有効 |
| 画像サイズを下げる | 最も効きやすい | 品質・構図も変わる |
| steps を下げる | 時間短縮 | 品質が落ちる場合がある |
| 使わない pipeline を削除 | メモリ解放 | `del` と `gc.collect()` を使う |


In [ ]:
def clear_memory(*objects):
    for obj in objects:
        del obj
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


if torch.cuda.is_available():
    print(torch.cuda.memory_summary(device=None, abbreviated=True))


## 15. 生成結果を評価する

生成画像の評価は、単に「きれいか」だけでは不十分です。用途に合わせて観点を分けます。

| 観点 | 確認すること |
|---|---|
| prompt 忠実度 | 主体、属性、構図が指示通りか |
| 破綻 | 手、文字、反射、左右対称、境界の不自然さ |
| 一貫性 | シリーズ画像で画風・色・構図が揃うか |
| 実用性 | 後工程の切り抜き、編集、掲載サイズに耐えるか |
| リスク | 著作権、商標、人物の権利、センシティブ表現 |

実務では生成結果だけでなく、設定も CSV などで残します。

In [ ]:
experiment_log = pd.DataFrame([
    {
        'name': 'base',
        'model_id': model_id,
        'prompt': base_config.prompt,
        'negative_prompt': base_config.negative_prompt,
        'seed': base_config.seed,
        'steps': base_config.steps,
        'guidance_scale': base_config.guidance_scale,
        'width': base_config.width,
        'height': base_config.height,
        'file': 'text2img_base.png',
    }
])

log_path = output_dir / 'experiment_log.csv'
experiment_log.to_csv(log_path, index=False)
experiment_log


## 16. 演習

1. `base_config.prompt` の主体だけを変え、同じ seed で比較してください。
2. `guidance_scale` を 3, 7.5, 13 に変え、prompt 忠実度と破綻を記録してください。
3. `EulerDiscreteScheduler` と `DPMSolverMultistepScheduler` で、20 steps の結果を比較してください。
4. img2img の `strength` を 0.2, 0.5, 0.8 に変え、元画像の保持具合を評価してください。
5. inpainting の mask を小さくした場合と大きくした場合で、周辺とのなじみ方を比較してください。

発展課題:

- 商品画像、建築パース、キャラクター、UI モックなど、1つの用途を決めて prompt テンプレートを作る。
- 生成画像ごとに評価点を付け、設定と結果の関係を分析する。
- ControlNet や IP-Adapter を使い、構図や参照画像の制御を追加する。

## まとめ

Stable Diffusion を実践で使うときは、モデルだけでなく、prompt、negative prompt、seed、steps、guidance scale、scheduler、画像サイズ、編集方法をまとめて扱います。

重要な考え方:

- seed は構図や細部の出発点を決める。
- guidance scale は prompt への従いやすさと不自然さのトレードオフを持つ。
- steps は品質と速度のトレードオフを持つ。
- scheduler は同じ条件でも画作りを変える。
- img2img と inpainting は、ゼロから生成するより実務的な編集に向いている。
- 良い結果を再利用するには、画像だけでなく生成設定を必ず保存する。